# Nested Cross-Validation — evaluating a procedure, not a model

## The problem, one more turn of the screw

The first notebook made this point: the score you use to *choose* a model is optimistic,
because you selected the maximum of several noisy numbers. The fix was a held-out test set.

Here is the part that fix does not cover.

A single held-out test set gives you **one** number, from **one** split. We spent the whole
Bias-Variance lesson establishing that one split is a noisy estimate — and then, the moment
we started tuning, went straight back to relying on one.

Worse: the tuning itself depends on that split. A different train/test division would produce
a different training set, a different winning hyperparameter, *and* a different test score.
When you report "we tuned `C` and got 92.9%", you are reporting a property of one particular
split, not of your method.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 42

penguins = pd.read_csv("data/penguins.csv").dropna(subset=["sex"])

X = penguins.drop(columns="sex")
y = penguins["sex"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RSEED, stratify=y
)

numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy="most_frequent"),
                            OneHotEncoder(handle_unknown="ignore"))

preprocessor = make_column_transformer(
    (numeric, make_column_selector(dtype_include="number")),
    (categorical, make_column_selector(dtype_exclude="number")),
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

print(f"{len(X_train)} training penguins, {len(X_test)} held out")

## Two loops

**Nested cross-validation** puts the entire tuning procedure inside a cross-validation loop.

```
outer fold 1:  ├── inner CV over the training part → pick best params → fit → score on the outer test part
outer fold 2:  ├── inner CV over the training part → pick best params → fit → score on the outer test part
outer fold 3:  ├── …
outer fold 4:  ├── …
outer fold 5:  └── …
                   ↓
              five scores, from five independently tuned models
```

- The **inner** loop selects hyperparameters. It only ever sees the outer training part.
- The **outer** loop scores. Its test part is untouched by the selection that produced the
  model being scored.

The output is not a model. It is an estimate of **how well this whole procedure performs** —
"logistic regression, with `C` chosen by 5-fold search" as a single object.

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold, cross_validate

param_grid = {"classifier__C": [0.01, 0.1, 1, 10, 100]}

inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)

# the tuner is itself an estimator — that is what makes nesting possible
tuner = GridSearchCV(model, param_grid, cv=inner_cv, scoring="accuracy", n_jobs=-1)

nested = cross_validate(tuner, X_train, y_train, cv=outer_cv,
                        scoring="accuracy", return_estimator=True)

print("outer fold scores:", np.round(nested["test_score"], 4))
print(f"nested estimate  : {nested['test_score'].mean():.4f} ± {nested['test_score'].std():.4f}")

That interval is the honest description of the procedure.

Compare the three numbers we now have for the same work:

In [ ]:
flat = GridSearchCV(model, param_grid, cv=inner_cv, scoring="accuracy", n_jobs=-1)
flat.fit(X_train, y_train)

print(f"1. best_score_ from a single search  : {flat.best_score_:.4f}   ← optimistic: the max of 5 noisy scores")
print(f"2. score on one held-out test set    : {flat.score(X_test, y_test):.4f}   ← unbiased, but from one split")
print(f"3. nested cross-validation           : {nested['test_score'].mean():.4f} ± {nested['test_score'].std():.4f}   ← unbiased, with a spread")

**Only the third comes with an error bar**, and the error bar is the point. Without it you
cannot say whether a rival method scoring 0.94 is genuinely better or one lucky split away
from yours.

## Did the folds even agree?

`return_estimator=True` keeps each outer fold's fitted tuner, so you can see what each one
chose.

In [ ]:
for i, est in enumerate(nested["estimator"], start=1):
    print(f"outer fold {i}: chose C = {est.best_params_['classifier__C']:<6} "
          f"(inner cv {est.best_score_:.4f}) → outer score {nested['test_score'][i-1]:.4f}")

If the folds disagree about the best `C`, that is not a bug — it is the finding. It means
the choice is not well determined by this much data, and reporting a single "optimal" value
would be overstating what you know.

## The cost

Nested cross-validation multiplies: `outer folds × inner folds × combinations` fits.
Ours is 5 × 5 × 5 = 125, which is instant on 250 penguins. On a real dataset with a serious
search space it can be the difference between minutes and a day.

In [ ]:
outer, inner, combos = 5, 5, len(param_grid["classifier__C"])
print(f"plain search : {inner * combos:>5} fits")
print(f"nested       : {outer * inner * combos:>5} fits   ({outer}× more)")

## So which do I actually use?

The two loops answer different questions, and you usually want both:

| Question | Use |
|---|---|
| *How good is this approach?* — a number for a report, or a comparison against another method | **nested CV** |
| *Which settings should I ship?* — the actual model that goes into production | **a single search on all your data**, then take `best_estimator_` |

They are not alternatives. Run nested CV to get an honest estimate, then run one final search
on everything to build the model you deploy. The nested estimate describes what you should
expect from it.

## Summary

- `best_score_` is optimistic. A single test score is unbiased but noisy.
- **Nested cross-validation** wraps the whole tuning procedure in an outer CV loop, giving an
  estimate **with a spread**.
- Inspect what each outer fold chose. Disagreement means the hyperparameter is not pinned
  down by your data.
- It costs `outer × inner × combinations` fits — budget for it.

> **Where this goes next.** The **Advanced ML-Pipelines** lesson searches over *models* and
> *preprocessing choices*, not just their settings — and everything here applies unchanged,
> because a search object is just another estimator.